In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from keras import layers, models, optimizers
from keras.utils import plot_model
from IPython.display import Image
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

In [ ]:
dataset_csv_path = '/content/drive/MyDrive/MachineLearningCVE/'
csv_file_names = [
    'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
    'Friday-WorkingHours-Morning.pcap_ISCX.csv',
    'Monday-WorkingHours.pcap_ISCX.csv',
    'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
    'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
    'Tuesday-WorkingHours.pcap_ISCX.csv',
    'Wednesday-workingHours.pcap_ISCX.csv'
]

full_path = [os.path.join(dataset_csv_path, csv_file) for csv_file in csv_file_names]
df = pd.concat((pd.read_csv(file) for file in full_path), ignore_index=True)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.describe()

In [ ]:
# Print columns with null values
null_columns = df.columns[df.isnull().any()]
print("Columns with null values:")
print(df[null_columns].isnull().sum())

# Print columns with identical values
identical_columns = df.columns[df.apply(lambda x: x.nunique() == 1)]
print("\nColumns with identical values:")
print(identical_columns)

In [ ]:
# Drop columns with null values
df = df.drop(columns=null_columns)

# Drop columns with identical values
df = df.drop(columns=identical_columns)

In [ ]:
df.shape

In [ ]:
# Print duplicate rows
duplicate_rows = df[df.duplicated()]
print("\nDuplicate rows:")
print(len(duplicate_rows))

In [ ]:
# Drop duplicate rows
df = df.drop_duplicates()

In [ ]:
df.shape

In [ ]:
# Histogram of 'Flow Duration'
plt.figure(figsize=(10, 6))
sns.histplot(df[' Flow Duration'], bins=30, kde=True)
plt.title('Distribution of Flow Duration')
plt.xlabel('Flow Duration')
plt.ylabel('Frequency')
plt.show()

In [ ]:
#Bar plot of 'Label'
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x=' Label')
plt.title('Distribution of Classes')
plt.xlabel('Class Label')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Bar plot of the top 10 most frequent values in 'Destination Port'
top_ports = df[' Destination Port'].value_counts().head(10)
plt.figure(figsize=(10, 6))
sns.barplot(x=top_ports.index, y=top_ports.values, palette='viridis')
plt.title('Top 10 Most Frequent Destination Ports')
plt.xlabel('Destination Port')
plt.ylabel('Frequency')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Perform label encoding on 'Label' column
label_encoder = LabelEncoder()
df['Encoded_Label'] = label_encoder.fit_transform(df[' Label'])

In [ ]:
# Calculate correlations between features and encoded label
correlations = df.corr()['Encoded_Label'].abs().sort_values(ascending=False)

# Plot top 10 most correlated features in a bar plot
plt.figure(figsize=(10, 6))
top_correlated_features = correlations[1:20]  # Exclude the label itself
top_correlated_features.plot(kind='bar', color='skyblue')
plt.title('Top 10 Most Correlated Features with Encoded Label')
plt.xlabel('Feature')
plt.ylabel('Absolute Correlation')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Select the top 10 most correlated features with the encoded label
top_correlated_features_names = top_correlated_features.index.tolist()

# Create a new DataFrame with 'Label' and the top 10 most correlated features
correlated_df = df[[' Label'] + top_correlated_features_names[:20]]  # Select only the first 15 features

# Display the first few rows of the new DataFrame
correlated_df.head()

In [ ]:
# merging similar classes with low instances
correlated_df[" Label"] = correlated_df[" Label"].replace(["Web Attack � Brute Force","Web Attack � XSS","Web Attack � Sql Injection"],"Web Attack")

correlated_df[' Label'] = correlated_df[' Label'].replace({
    'Bot': 'Other',
    'Infiltration': 'Other',
    'Heartbleed': 'Other'
})

correlated_df[' Label'] = correlated_df[' Label'].replace({
    'DoS slowloris': 'DoS Attack',
    'DoS Slowhttptest': 'DoS Attack'
})
# Determine the size of the smallest class
correlated_df[' Label'].value_counts()


In [ ]:
class_names = correlated_df[' Label'].unique().tolist()
class_names

In [ ]:
correlated_df.head()

In [ ]:
# Perform label encoding on 'Label' column
label_encoder = LabelEncoder()
correlated_df['Encoded_Label'] = label_encoder.fit_transform(correlated_df[' Label'])

In [ ]:
correlated_df.head()

In [ ]:
# Separate features and labels
X = correlated_df.drop([' Label', 'Encoded_Label'], axis=1)
y = correlated_df['Encoded_Label']

# Preprocess the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the data into training and test sets
X_train_all, X_test, y_train_all, y_test = train_test_split(X_scaled, y, test_size=0.2,stratify=y, random_state=42)

# Split data into labeled and unlabeled sets
X_labeled, X_unlabeled, y_labeled, _ = train_test_split(X_train_all, y_train_all, test_size=0.9, stratify=y_train_all, random_state=42)

In [ ]:
# Build the model
def build_model():
    # Input layer
    input_layer = layers.Input(shape=(X.shape[1],))

    # Branch 1: Convolutional layers
    conv1d_output = layers.Reshape((X.shape[1], 1))(input_layer)
    conv1d_output = layers.Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(conv1d_output)
    conv1d_output = layers.MaxPooling1D(pool_size=2)(conv1d_output)
    conv1d_output = layers.Conv1D(filters=128, kernel_size=3, activation='relu', padding='same')(conv1d_output)
    conv1d_output = layers.MaxPooling1D(pool_size=2)(conv1d_output)
    conv1d_output = layers.Conv1D(filters=256, kernel_size=3, activation='relu', padding='same')(conv1d_output)
    conv1d_output = layers.MaxPooling1D(pool_size=2)(conv1d_output)
    conv1d_output = layers.Flatten()(conv1d_output)

    # Branch 2: GRU layers
    gru_output = layers.Reshape((1, X.shape[1]))(input_layer)
    gru_output = layers.Permute((2, 1))(gru_output)
    gru_output = layers.GRU(64, return_sequences=True)(gru_output)
    gru_output = layers.GRU(64, return_sequences=True)(gru_output)
    gru_output = layers.GRU(64)(gru_output)

    # Concatenate outputs from Branch 1 and Branch 2
    concatenated_output = layers.Concatenate()([conv1d_output, gru_output])

    # Fully connected layers
    fc1 = layers.Dense(128, activation='relu')(concatenated_output)
    dropout = layers.Dropout(0.5)(fc1)
    output_layer = layers.Dense(10, activation='softmax')(dropout)

    # Define model
    model = models.Model(inputs=input_layer, outputs=output_layer)
    return model


# Instantiate the model
model = build_model()

# Compile the model
optimizer = optimizers.Adam()
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Plot the model architecture
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True)

Image(filename='model_plot.png')

In [ ]:
# Train the model (semi-supervised learning)
batch_size = 64
num_batches_labeled = len(X_labeled) // batch_size
num_batches_unlabeled = len(X_unlabeled) // batch_size
num_batches = min(num_batches_labeled, num_batches_unlabeled)

# Initialize lists to store accuracy and loss values
accuracy_values = []
loss_labeled_values = []
loss_unlabeled_values = []

for epoch in range(5):
    print(f"Epoch {epoch + 1}/{5}")
    total_loss_labeled = []
    total_loss_unlabeled = []
    total_accuracy = 0
    for batch in range(num_batches):
        # Train with labeled data
        labeled_start = batch * batch_size
        labeled_end = (batch + 1) * batch_size
        X_labeled_batch = X_labeled[labeled_start:labeled_end]
        y_labeled_batch = y_labeled[labeled_start:labeled_end]
        loss, accuracy = model.train_on_batch(X_labeled_batch, y_labeled_batch)
        total_loss_labeled.append(loss)
        total_accuracy += accuracy

        # Train with unlabeled data (pseudo-labeling)
        unlabeled_start = batch * batch_size
        unlabeled_end = (batch + 1) * batch_size
        X_unlabeled_batch = X_unlabeled[unlabeled_start:unlabeled_end]
        pseudo_labels = np.argmax(model.predict(X_unlabeled_batch), axis=1)

        # Ensure pseudo-labels have the correct shape
        pseudo_labels = pseudo_labels.reshape((-1, 1))  # Reshape to match model output

        # Train the model with the pseudo-labeled data
        loss = model.train_on_batch(X_unlabeled_batch, pseudo_labels)
        total_loss_unlabeled.append(loss)

    avg_loss_labeled = np.mean(total_loss_labeled)
    avg_loss_unlabeled = np.mean(total_loss_unlabeled)
    avg_accuracy = total_accuracy / num_batches

    # Append accuracy and loss values to lists
    accuracy_values.append(avg_accuracy)
    loss_labeled_values.append(avg_loss_labeled)
    loss_unlabeled_values.append(avg_loss_unlabeled)

    print(f"Loss (labeled): {avg_loss_labeled}, Loss (unlabeled): {avg_loss_unlabeled}, Accuracy: {avg_accuracy}")

In [ ]:
# Plot accuracy
plt.plot(range(1, 6), accuracy_values, label='Accuracy', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy During Training')
plt.legend()
plt.show()

# Plot loss
plt.plot(range(1, 6), loss_labeled_values, label='Loss (labeled)', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss During Training labeled')
plt.legend()
plt.show()

# Plot loss
plt.plot(range(1, 6), loss_unlabeled_values, label='Loss (unlabeled)', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss During Training unlabeled')
plt.legend()
plt.show()

In [ ]:
model.save('model.h5')

In [ ]:
pred = model.predict(X_test)

In [ ]:
predicted_labels = np.argmax(pred, axis=1).reshape(-1, 1)

# Generate the classification report
report = classification_report(y_test , predicted_labels, target_names=class_names)

print("Classification Report:")
print(report)

In [ ]:
# Generate the confusion matrix
conf_matrix = confusion_matrix(y_test , predicted_labels)

# Plot the confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.show()